# Stock Price Prediction — Hybrid ARIMA-LSTM (Demo)

This notebook walks through the full pipeline step by step:
1. Load & clean data
2. Stationarity check (ADF) + differencing
3. Train/test split & scaling
4. AR / MA / ARMA / ARIMA (walk-forward)
5. LSTM
6. Hybrid ARIMA-LSTM
7. Compare metrics

> Requires: `pip install -r requirements.txt` (numpy, pandas, scikit-learn, statsmodels, tensorflow, matplotlib).

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))
import numpy as np, pandas as pd, matplotlib.pyplot as plt

from src import data_preprocessing as dp
from src import arima_models as am
from src import evaluate as ev
from src import utils
utils.set_seed(42)

## 1. Load & clean

In [ ]:
df = dp.load_data('../data/sample_stock_data.csv')
df = dp.clean_data(df)
series = dp.get_target_series(df, 'Close')
print(series.describe())
series.plot(figsize=(12,4), title='Close price'); plt.show()

## 2. Stationarity (ADF) & differencing

In [ ]:
print('Raw:', dp.adf_test(series))
d = dp.find_differencing_order(series)
print('Chosen d =', d)
print('Differenced:', dp.adf_test(dp.difference(series, d)))

## 3. Train/test split

In [ ]:
train, test = dp.train_test_split_series(series, 0.8)
train_vals, test_vals = train.values, test.values
print('Train', len(train), 'Test', len(test))
metrics, preds = {}, {}

## 4. AR / MA / ARMA / ARIMA (walk-forward)

In [ ]:
arima_order = am.auto_order(train_vals, d=d)
print('ARIMA order:', arima_order)
orders = {'AR':(2,0,0), 'MA':(0,0,2), 'ARMA':(2,0,2), 'ARIMA':arima_order}
for name, order in orders.items():
    yhat = am.rolling_forecast(train_vals, test_vals, order)
    metrics[name] = ev.print_metrics(name, test_vals, yhat)
    preds[name] = yhat

## 5. Standalone LSTM

In [ ]:
from src import lstm_model as lm
lm.set_tf_seed(42)
look_back = 30
tr_s, te_s, scaler = dp.scale_train_test(train_vals, test_vals)
full = np.concatenate([tr_s.ravel(), te_s.ravel()])
X_tr, y_tr = lm.create_sequences(tr_s.ravel(), look_back)
model = lm.build_lstm(look_back); lm.train_lstm(model, X_tr, y_tr, epochs=30, verbose=0)
n = len(tr_s); ps = []
for t in range(len(te_s)):
    w = full[n+t-look_back:n+t].reshape(1, look_back, 1)
    ps.append(lm.predict_lstm(model, w)[0])
lstm_pred = dp.inverse_scale(scaler, np.array(ps))
metrics['LSTM'] = ev.print_metrics('LSTM', test_vals, lstm_pred); preds['LSTM'] = lstm_pred

## 6. Hybrid ARIMA-LSTM

In [ ]:
from src.hybrid_model import hybrid_arima_lstm
res = hybrid_arima_lstm(train_vals, test_vals, arima_order=arima_order,
                        look_back=30, epochs=30, verbose=0)
metrics['ARIMA-LSTM'] = ev.print_metrics('ARIMA-LSTM', test_vals, res['hybrid'])
preds['ARIMA-LSTM'] = res['hybrid']

## 7. Compare

In [ ]:
plt.figure(figsize=(13,6))
plt.plot(test_vals, 'k', label='Actual', lw=1.6)
for name in ['ARIMA','LSTM','ARIMA-LSTM']:
    if name in preds: plt.plot(preds[name], label=name, alpha=.85)
plt.legend(); plt.title('Forecast vs Actual'); plt.grid(alpha=.3); plt.show()

pd.DataFrame(metrics).T[['RMSE','MAE','MAPE','R2']].round(4)